# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import pandas
import datetime
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

# Values to edit

In [2]:
site_names = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
targets = ["Seagrass", "Ulva", "Gracilaria"]

# Cells to run

In [3]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:58400/status,
Dashboard: http://127.0.0.1:58400/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:58401,Workers: 0
Dashboard: http://127.0.0.1:58400/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:58426,Total threads: 2
Dashboard: http://127.0.0.1:58427/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:58404,


In [4]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"
website_path = data_path / "website" / "uav_areas"

In [10]:
for uav_site in site_names:
    print(f"Creating UAV target area files for {uav_site}")
    output_file = website_path / uav_site / "uav_areas.csv"
    output_file.parent.mkdir(exist_ok=True, parents=True)
    if output_file.exists():
        print(f"{uav_site} area's already recorded. Skip.")
        #continue

    target_areas = {key: [] for key in ["date"] + targets}
    
    training_labels = pandas.read_csv(
        training_labels_file, sep="\t", header=None, names=["Value", "Key"]
    ).set_index('Key')['Value'].to_dict()
    survey_dates = pandas.read_csv(servey_dates_file)
    survey_date = datetime.datetime.strptime(
        survey_dates[survey_dates["site"] == uav_site]["date"].iloc[0],
        sentinel2.DATE_FORMAT_SITE_SURVEY
    ).date()
    
    target_areas["date"].append(survey_date)

    uav_file = uav_folder / f"{uav_site}_classified.tif"
    uav_data = utils.load_classification(filename=uav_file, chunks=True)
    pixel_area = max(uav_data.rio.resolution())**2

    for target in targets:
        target_id = training_labels[target]
        target_area = float(((uav_data == target_id).sum() * pixel_area).compute())
        target_areas[target].append(target_area)
    target_areas = pandas.DataFrame(target_areas)
    target_areas.to_csv(output_file, index=False)

Creating UAV target area files for CatlinsLake
CatlinsLake area's already recorded. Skip.
Creating UAV target area files for CatlinsRiverMouth
CatlinsRiverMouth area's already recorded. Skip.
Creating UAV target area files for Childrens
Childrens area's already recorded. Skip.
Creating UAV target area files for Duvauchelle
Duvauchelle area's already recorded. Skip.
Creating UAV target area files for Robinsons
Robinsons area's already recorded. Skip.
Creating UAV target area files for Takamatua
Takamatua area's already recorded. Skip.
Creating UAV target area files for Purau
Purau area's already recorded. Skip.
Creating UAV target area files for Ihutai
Ihutai area's already recorded. Skip.
Creating UAV target area files for IveyBay_Nov25
IveyBay_Nov25 area's already recorded. Skip.
Creating UAV target area files for IveyBay_Feb26
IveyBay_Feb26 area's already recorded. Skip.
Creating UAV target area files for LeftBank_Nov25
LeftBank_Nov25 area's already recorded. Skip.
Creating UAV targe